Trail through the Cheyne Range

Modelling a new walking track along the Cheyne Range, Tasmania

Authors: Gabriel Pham, Sanden Cook  ·  KGG375 GIS Project (AT3), University of Tasmania, 2026

## 1. Introduction

### 1.1 Motivation
Tasmania is filled with beautiful scenery an untouched wilderness. This includes the Cheyne Range, located in the Central Highlands, it includes some of Tasmania's highest peaks. This is a well known walking track but is currently entirely off-track making it only for experienced bush-walkers. 

![Walkers on the Cheyne Range](Cheyne%20Range.jpg)

*Walkers on the Cheyne Range. Photo: © Louise Fairfax.*

### 1.2 Background
Off-track walking in places like the Cheyne Range creates informal trails called "pads". These tend to widen and erode over time especially on fragile alpine vegetation. A planned hardened track can protect the surrounding landscape. Deciding where to put this track requires the weighing of many environmental factors at once such as slope and vegetation type. MCDA is currently the standard way to combine environmental factors into a suitability map in which we will be using WLC and TOPSIS. To weight the criteria we used AHP.

### 1.3 Objectives
The aim of this project is to model and compare optimal trails that go through the Cheyne range which weighs all the environmental factors to find the best fit. 

Objectives:
Develop and compare two suitability models for a walking track using different decision rules (WLC and TOPSIS)
Implement a movement cost model that accounts for the effort of uphill and downhill walking.
Generate and compare routes from different start points using a least-cost path algorithm.
Compare and visualise the resulting routes using maps and elevation profiles. 

## 2. Study Area and Data

### 2.1 Study area


![Topographic map of the Cheyne Range study area](CheyneRangetopo.png)

*Figure 1: Topographic map of the Cheyne Range and surrounds. Base map: LISTmap, © State of Tasmania (CC BY 3.0 AU).*


### 2.2 Data sources

All datasets were clipped to a 17.5 × 13.5 km study area (the start, end and summit points plus a 1 km buffer), projected to GDA2020 / MGA Zone 55 (EPSG:7855) and converted to a common 10 m grid.

| Dataset | Source | Type / resolution | Used for |
|---|---|---|---|
| Tasmania Statewide 2 m Digital Elevation Model | Mineral Resources Tasmania / Land Tasmania, via theLIST | Raster, 2 m (resampled to 10 m by cell averaging) | Slope criterion and constraint, movement cost, elevation profiles |
| TASVEG 5.0 vegetation communities | Tasmanian Vegetation Monitoring and Mapping Program, NRE Tas, via theLIST open data service | Vector polygons, 1:25 000 nominal scale | Vegetation criterion, threatened community constraints, buttongrass (wet ground) |
| LIST Hydrographic Lines | Land Tasmania, via theLIST open data service | Vector lines | Distance-to-streams criterion |
| LIST Hydrographic Areas | Land Tasmania, via theLIST open data service | Vector polygons | Lakes and tarns (constraint), swamps and wet areas (wetland criterion), wide rivers (streams criterion) |
| LIST Transport Segments | Land Tasmania, via theLIST open data service | Vector lines | Distance-to-access criterion (roads and walking tracks) |
| Start/end access points and summits | Created by the authors in ArcGIS Pro from road ends, track heads and named peaks | Vector points | Route start, end and waypoints |

Threatened vegetation communities were identified by matching TASVEG 5.0 communities against Schedule 3A of the *Nature Conservation Act 2002* (Tas).

**Data limitations:**
- Approximately 65% of the statewide DEM is derived from LiDAR. The remainder is interpolated from photogrammetric contours and spot heights, with an effective resolution of 10 m
- TASVEG and threatened community mapping are indicative only and would require field verification.

LIST data © State of Tasmania, licensed under Creative Commons Attribution 3.0 Australia.

## 3. Methods

### 3.1 Workflow overview
*A flow chart of the full workflow: data → criteria → suitability → movement cost → routes → sensitivity. List the Python libraries used.*

![Workflow flow chart](workflow_flowchart.png)

*Figure 2: Overview of the analysis workflow.*

### 3.2 Suitability criteria and constraints
The criteria is scored 0 to 1 with 1 being ideal.
Slope: derived from the DEM starts at 1 up to 10 degrees then linearly falls to 0 at 30 degrees. 
Vegetation: derived from TASVEG 5.0, this is scored by individual vegetation groups. 
    Modified land: 1.0
    Dry Eucalypt Forest and woodland: 0.9
    Native Grassland: 0.8
    Other natural environments (rocks): 0.7
    Wet Eucalypt Forest, non-Eucalypt forest: 0.6
    Moorland, rushland, buttongrass: 0.5
    Highland treeless vegetation: 0.4
    Scrub: 0.3
    Wetland: 0.2
    Alpine Coniferous Heathland: 0.1
Streams: Derived from hydrographic lines and areas. 0 within 30m of water and rising to 1 at 100m. 
Wetland: Derived from hydrographic areas (swamps, buttongrass). 
Access: Derived from transport segments. 1 on a road or track. 

Constraints:
Lakes and Tarns: Track can not go over open water
Pencil Pine and King Billy Pine: Threatened Vegetation
Sphagnum Peatland: Threatened Vegetation
Cushioned Moorland, Highland Poa Grassland, Subalpine rushland: threatened Vegetation
Steep Ground: Slope > 35 degrees

### 3.3 Criteria weights (AHP)
The comparison matrix we chose to use is AHP. This uses a table with all the criteria ranked against each other on a scale of 1-9. It describes how much more important a row criterion is than the column criterion on Saaty's Scale. On the scale 1 means equal, 3 is moderate and 5 is strong. The scale goes up to 9 which is extreme but none of our criteria needs to be that strong over another. For example, Slope vs. Access, Slope is given a 5 over access because access is only a convenience while slope decides sustainability as well as how well the trail can be walked.
The script then takes this table and scales it to sum to 1.
Slope: 0.407
Vegetation: 0.249
Streams: 0.143
Wetland: 0.143
Access: 0.059

| Criterion | Slope | Vegetation | Streams | Wetland | Access | **Weight** |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| **Slope** | 1 | 2 | 3 | 3 | 5 | **0.407** |
| **Vegetation** | 1/2 | 1 | 2 | 2 | 4 | **0.249** |
| **Streams** | 1/3 | 1/2 | 1 | 1 | 3 | **0.143** |
| **Wetland** | 1/3 | 1/2 | 1 | 1 | 3 | **0.143** |
| **Access** | 1/5 | 1/4 | 1/3 | 1/3 | 1 | **0.059** |


### 3.4 Suitability models (WLC and TOPSIS)
Two decision rules were used to combine the five standardised criteria into two suitability rasters. Both use the same criteria and the same AHP weights, this is so that any difference is because of the rules alone.

Weighted Linear Combination (WLC)
This is the most widely used decision rule in GIS-based multicriteria analysis, with each cell's suitability being the sum of its criterion scores multiplied by their weights.

TOPSIS
This ranks each option by how close each cell is to an ideal solution and how far it is from a worst-case solution.


### 3.5 movement cost

movement cost was from a simple hiking cost fourmula where slope cost was based on the degrees*0.1 where slope was positive eg. 30 degrees has a cost weight of 0.3 and for negitive slopes the cost was assigned degrees*0.01 which makes downhill impact walking speed/effort less

importanly these implement a direction modifier which calculates the walkers apparrent slope rather then the global slope values, making this a anisotropic cost surface.


### 3.6 Path optimisation

for path optimisation Dijkstra algorithm was used, this finds the shortest path by slowly extending the, search and prioritising the least cost path to each node

due to the limitations of python and data storage the anisotropic cost surface does not exist in iteslf instead it is defined in the least cost path algorithm process, where the pathfinding can input the direction and calculate the cost.



## 4. Results

### 4.1 Suitability surfaces

![WLC and TOPSIS suitability maps](suitability_maps.png)

*Figure 3: Suitability for a walking track under WLC (left) and TOPSIS (right). White areas are excluded by constraints.*


### 4.2 Routes
![Topographic map of the Cheyne Range study area](map_with_table.png)

*Figure 1: Topographic map with routes shaded by cost. Base map: LISTmap, © State of Tasmania (CC BY 3.0 AU).*




## 5. Discussion


### 5.1 Main hurdles and problems

The main hurdle on the criteria and suitability side of the project was deciding what to use and how it should be weighted. This came from looking to past reasearch and finding threatened vegetation and past modelling / pathfinding literature. 


### 5.2 Alternative approaches
Alternative approaches include using different criteria and contraints. We chose the ones that we thought to be most important and that were backed up by literature. It can be made more comprehensive if we added more criteria into the suitability. 

Another approach could also be to use a different weighting method. In this project we chose to use AHP as it allows the criteria to be ranked against eachother and then the weights are pulled from that. The issue with this is that it is still subjective and requires the criteria to be ranked apprprialty. 

The way the suitability score is calculated can also be changed. In this project we chose to make two rasters using WLC and TOPSIS. This was because these two methods are what we are familiar with. A different method to producing a suitability raster could be used which would handle the weights differently and produce an alternative raster. 

### 5.4 Limitations


## 6. Conclusion



## References

Dixon, G., Hawes, M., & McPherson, G. (2004). Monitoring and modelling walking track impacts in the Tasmanian Wilderness World Heritage Area, Australia. *Journal of Environmental Management, 71*(4), 305–319.

Hwang, C. L., & Yoon, K. (1981). *Multiple attribute decision making: Methods and applications*. Springer.

Malczewski, J. (2006). GIS-based multicriteria decision analysis: A survey of the literature. *International Journal of Geographical Information Science, 20*(7), 703–726.

Olive, N. D., & Marion, J. L. (2009). The influence of use-related, environmental, and managerial factors on soil loss from recreational trails. *Journal of Environmental Management, 90*(3), 1483–1493.

Rees, W. G. (2004). Least-cost paths in mountainous terrain. *Computers & Geosciences, 30*(3), 203–209.

Saaty, T. L. (1980). *The analytic hierarchy process*. McGraw-Hill.

Whinam, J., & Chilcott, N. M. (2003). Impacts after four years of experimental trampling on alpine/sub-alpine environments in western Tasmania. *Journal of Environmental Management, 67*(4), 339–351.

### Data sources

Land Tasmania. (2026). *LIST Hydrographic Areas* [Data set]. Department of Natural Resources and Environment Tasmania, theLIST. Retrieved October 2, 2026. © State of Tasmania, CC BY 3.0 AU.

Land Tasmania. (2026). *LIST Hydrographic Lines* [Data set]. Department of Natural Resources and Environment Tasmania, theLIST. Retrieved October 2, 2026. © State of Tasmania, CC BY 3.0 AU.

Land Tasmania. (2026). *LIST Transport Segments* [Data set]. Department of Natural Resources and Environment Tasmania, theLIST. Retrieved October 2, 2026. © State of Tasmania, CC BY 3.0 AU.

Land Tasmania. (2026). *Tasmania 1 m Digital Elevation Model* [Data set]. theLIST. © State of Tasmania, CC BY 3.0 AU. [Check exact name and source]

Tasmanian Vegetation Monitoring and Mapping Program. (2025). *TASVEG 5.0* [Data set]. Department of Natural Resources and Environment Tasmania, theLIST. Retrieved October 2, 2026. CC BY 3.0 AU.